In [1]:
import carla
import numpy as np
import cv2
import time


In [2]:
CARLA_HOST = "localhost"
CARLA_PORT = 2000
CARLA_TIMEOUT = 10.0

IMAGE_WIDTH = 1280
IMAGE_HEIGHT = 720
CAMERA_POS_Z = 3
CAMERA_POS_X = 0

In [3]:
def connect_to_carla() -> tuple[carla.Client, carla.World]:

    client = carla.Client(CARLA_HOST, CARLA_PORT)
    client.set_timeout(CARLA_TIMEOUT)
    world = client.get_world()

    if not world:
        raise RuntimeError(
            "Failed to return a valid world\n"
        )

    return client, world


In [4]:
def define_spawn_point(world: carla.World) -> carla.Transform:

    spawn_points = world.get_map().get_spawn_points()

    spawn_point = spawn_points[0]

    return spawn_point

In [5]:
def spawn_ego_vehicle(world: carla.World, spawn_point: carla.Transform) -> carla.Vehicle:

    vehicle_bp = world.get_blueprint_library().filter("vehicle.*")

    vehicle =  world.try_spawn_actor(
        vehicle_bp[0],
        spawn_point
    )

    return vehicle

In [6]:
def ego_vehicle_autopilot(vehicle: carla.Vehicle, autopilot_status: bool):

     vehicle.set_autopilot(
            autopilot_status,
        )
        

In [7]:
def spawn_depth_camera(world: carla.World, vehicle: carla.Vehicle) -> carla.Sensor:

    depth_sensor_bp = world.get_blueprint_library().filter("sensor.camera.depth")

    if not depth_sensor_bp:
        raise RuntimeError(
            "Carla was not able to return a valid depth sensor blueprint\n"
        )


    # sensor_transform =  carla.Transform(carla.Location(z = CAMERA_POS_Z, x = CAMERA_POS_X))
    # depth_sensor = world.spawn_actor(depth_sensor_bp, sensor_transform, attach_to=vehicle)
    
    return depth_sensor_bp

In [ ]:
def depth_callback(image, data_dict):

    image.convert(carla.ColorConverter.Depth)

In [8]:
def main():

    client, world = connect_to_carla()
    spawn_point = define_spawn_point(world)
    actors = world.get_actors()
    print("Destroying active actors\n")
    for actor in actors:
        actor.destroy()
    time.sleep(5)
    vehicle = spawn_ego_vehicle(world, spawn_point)
    ego_vehicle_autopilot(vehicle, False)
    depth_camera = spawn_depth_camera(world, vehicle)
    print(depth_camera)


In [ ]:
if __name__ == "__main__":
    main()

Destroying active actors

[ActorBlueprint(id=sensor.camera.depth,tags=[sensor, camera, depth])]


: 